# 02 — Tahoe per-compound transcriptomic vectors

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/cellduet/blob/main/notebooks/02_tahoe_percompound.ipynb)

Streams the 1,026-shard `pseudobulk_differential_expression` parquet from `tahoebio/Tahoe-100M`, picks ~2,000 HVGs by per-gene LFC variance (filtered to expressed genes), aggregates per `(drug, cell_line)` by mean log2FoldChange, and produces two per-compound transcriptomic phenotype products:

- **`tahoe_pooled_per_compound.h5ad`**: 379 × 2,000-HVG matrix, mean across all 47 QC-pass cell lines per drug.
- **`tahoe_a549_per_compound.h5ad`**: A549-only subset (the cell line shared with CPJUMP1 for the B3c sanity arm).

Outputs persist to HF dataset `patrickjreed/cellduet-tahoe-percompound`.

**Compute footprint.** 1,026 shards × ~91 MB ≈ 93 GB streamed from HF. Each shard has ~4M rows = (1 cell_line × 1 plate × 64 drugs × 62,710 genes). Caches to Colab `/content/` (~100 GB ephemeral). Total wall-clock: ~25–35 min on Colab Free with 8-worker parallel download. After HVG filter + NaN drop, the in-memory tall frame is ~1.2 GB (categorical-encoded); aggregation outputs are sub-MB.

**Schema (verified empirically against shard 0).** Tahoe ships DESeq2 statistics already plate-matched to DMSO controls per (cell_line, plate). Columns: `drug`, `concentration`, `plate`, `Cell_Name_Vevo`, `gene_name`, `log2FoldChange`, `baseMean` (+ `lfcSE`, `stat`, `pvalue`, `padj`, `n_cells_trt`, `n_cells_ctrl`, `Cell_ID_*`). We use `log2FoldChange` directly; the controls are already absorbed.

Refs: `docs/datasets/tahoe-100m.md`, `docs/datasets/joint.md`.

## 1. Install + imports

In [ ]:
!pip install -q pandas pyarrow huggingface_hub anndata
!pip install -q --no-deps "git+https://github.com/PatrickJReed/cellduet.git@main"

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download, login
from tqdm.auto import tqdm

print("imports OK")

## 2. HF login + cache dir

The HF push at the end needs an `HF_TOKEN` Colab secret with **write** scope. Read-only fetches work without auth.

In [ ]:
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
    print("HF login: OK (Colab secret)")
except Exception as e:
    print(f"Colab secret not used ({type(e).__name__}); relying on local cache or anon HF reads")

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = Path("/content/drive/MyDrive/cellduet/cache")
except (ImportError, Exception) as e:
    print(f"Drive skipped ({type(e).__name__}); using local cache")
    CACHE = Path.home() / ".cache" / "cellduet"
CACHE.mkdir(parents=True, exist_ok=True)
print(f"cache: {CACHE}")

## 3. Compound manifest (drugs to retain)

The 379-drug manifest produced by notebook 01. Used here as a sanity check on which drugs should appear in the Tahoe data.

In [ ]:
manifest_path = hf_hub_download(
    "patrickjreed/cellduet-compound-manifest", "compound_manifest.parquet", repo_type="dataset"
)
manifest = pd.read_parquet(manifest_path)
tahoe_drugs = set(manifest["drug"])
print(f"manifest: {len(manifest)} drugs ({len(tahoe_drugs)} unique)")

## 4. Sample 16 shards for HVG picking

We pick HVGs from a stratified sample (every 64th shard across the 1,026 total) to cover diverse `(cell_line, plate)` combinations without paying for the full read. HVG criterion: `baseMean > 10` (filter expression noise), then top 2,000 by `std(log2FoldChange)` across the sampled `(drug, cell_line, plate)` rows.

In [ ]:
REPO = "tahoebio/Tahoe-100M"
N_SHARDS = 1026
KEEP_COLS = ["drug", "concentration", "plate", "Cell_Name_Vevo", "gene_name", "log2FoldChange", "baseMean"]


def shard_path(idx: int) -> str:
    return f"metadata/pseudobulk_differential_expression/train-{idx:05d}-of-01026.parquet"


def download_shard(idx: int) -> str:
    return hf_hub_download(REPO, shard_path(idx), repo_type="dataset")

In [ ]:
sample_indices = list(range(0, N_SHARDS, 64))
print(f"Sampling {len(sample_indices)} shards across the 1026 total")

with ThreadPoolExecutor(max_workers=4) as pool:
    sample_paths = list(tqdm(pool.map(download_shard, sample_indices), total=len(sample_indices)))

dfs = []
for p in tqdm(sample_paths, desc="reading sample"):
    dfs.append(pd.read_parquet(p, columns=KEEP_COLS))
sample_df = pd.concat(dfs, ignore_index=True)
print(f"sample rows: {len(sample_df):,}, drugs: {sample_df['drug'].nunique()}, cell_lines: {sample_df['Cell_Name_Vevo'].nunique()}")

In [ ]:
# Pick HVGs: baseMean > 10, top 2000 by std(log2FoldChange)
nonnan = sample_df.dropna(subset=["log2FoldChange"])
gene_stats = nonnan.groupby("gene_name").agg(
    mean_baseMean=("baseMean", "mean"),
    std_lfc=("log2FoldChange", "std"),
)
expressed = gene_stats[gene_stats["mean_baseMean"] > 10]
hvg = expressed.nlargest(2000, "std_lfc")
hvg_set = set(hvg.index)

print(f"genes with mean_baseMean > 10: {len(expressed):,}")
print(f"HVG selected: {len(hvg_set):,}")
print(f"top 10 HVG by LFC std:")
print(hvg.head(10).to_string())

del sample_df, dfs, nonnan
import gc
gc.collect()

## 5. Stream + filter all 1,026 shards

Parallel download (8 workers), per-shard read, filter to HVGs + drop NaN log2FC, cast strings to categorical to keep RAM bounded. Total filtered tall frame is on the order of 1.2 GB.

This is the longest cell in the notebook (~25–35 min on Colab Free). Progress bar shows shards completed.

In [ ]:
def filter_shard(idx: int) -> pd.DataFrame:
    p = download_shard(idx)
    d = pd.read_parquet(p, columns=KEEP_COLS)
    d = d[d["gene_name"].isin(hvg_set) & d["log2FoldChange"].notna()]
    return d[["drug", "Cell_Name_Vevo", "gene_name", "log2FoldChange"]]


with ThreadPoolExecutor(max_workers=8) as pool:
    chunks = list(tqdm(pool.map(filter_shard, range(N_SHARDS)), total=N_SHARDS, desc="shards"))

print(f"\nfiltered shards: {len(chunks)}")
print(f"total filtered rows: {sum(len(c) for c in chunks):,}")

## 6. Concat + aggregate per (drug, cell_line, gene)

In [ ]:
tall = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()

# Cast string columns to categoricals for memory
for col in ("drug", "Cell_Name_Vevo", "gene_name"):
    tall[col] = tall[col].astype("category")

print(f"tall shape: {tall.shape}, memory: {tall.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(f"unique drugs: {tall['drug'].nunique()}")
print(f"unique cell lines: {tall['Cell_Name_Vevo'].nunique()}")
print(f"unique HVGs: {tall['gene_name'].nunique()}")

In [ ]:
# Aggregate to per-(drug, cell_line, gene) by mean log2FC across plates+concentrations
agg = (
    tall.groupby(["drug", "Cell_Name_Vevo", "gene_name"], observed=True)
    ["log2FoldChange"].mean().reset_index()
)
print(f"aggregated rows: {len(agg):,}")
print(f"unique (drug, cell_line) tuples: {agg.groupby(['drug','Cell_Name_Vevo'], observed=True).ngroups:,}")

del tall
gc.collect()

## 7. Pivot to wide (drug × cell_line, HVG)

In [ ]:
wide = agg.pivot_table(
    index=["drug", "Cell_Name_Vevo"],
    columns="gene_name",
    values="log2FoldChange",
    observed=True,
)
print(f"wide shape: {wide.shape}")
# Drugs covered + cell lines covered
n_drugs = wide.index.get_level_values("drug").nunique()
n_lines = wide.index.get_level_values("Cell_Name_Vevo").nunique()
print(f"drugs: {n_drugs}/379, cell lines: {n_lines}")

## 8. Build Tahoe-pooled product

Mean across cell lines per drug. Drugs with at least one cell line covered will appear; the per-gene mean ignores NaN cell-line cells via pandas default.

In [ ]:
pooled = wide.groupby(level="drug", observed=True).mean()
print(f"Tahoe-pooled: {pooled.shape}")
print(f"NaN fraction (drugs missing some HVG): {pooled.isna().mean().mean()*100:.2f}%")
# Fill NaN with 0 (drugs with no cells expressing a given gene)
pooled = pooled.fillna(0).astype(np.float32)

## 9. Build Tahoe-A549 product

A549-only subset; this is the transcriptomic input to the B3c sanity arm (paired with CPJUMP1-A549 morphology in notebook 06).

In [ ]:
if "A549" in wide.index.get_level_values("Cell_Name_Vevo"):
    a549 = wide.xs("A549", level="Cell_Name_Vevo")
    a549 = a549.fillna(0).astype(np.float32)
    print(f"Tahoe-A549: {a549.shape}")
else:
    raise RuntimeError("A549 not present in wide. Check upstream filtering.")

## 10. Save as h5ad

In [ ]:
def to_anndata(df: pd.DataFrame) -> ad.AnnData:
    obs = pd.DataFrame(index=df.index.astype(str))
    var = pd.DataFrame(index=df.columns.astype(str))
    return ad.AnnData(X=df.values.astype(np.float32), obs=obs, var=var)


pooled_ad = to_anndata(pooled)
a549_ad = to_anndata(a549)

pooled_path = CACHE / "tahoe_pooled_per_compound.h5ad"
a549_path = CACHE / "tahoe_a549_per_compound.h5ad"
pooled_ad.write_h5ad(pooled_path)
a549_ad.write_h5ad(a549_path)
print(f"saved {pooled_path}  ({pooled_path.stat().st_size/1024:.1f} KB)")
print(f"saved {a549_path}  ({a549_path.stat().st_size/1024:.1f} KB)")

## 11. (optional) Push to HF

In [ ]:
try:
    api = HfApi()
    repo_id = "patrickjreed/cellduet-tahoe-percompound"
    api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)
    for path in (pooled_path, a549_path):
        api.upload_file(
            path_or_fileobj=str(path),
            path_in_repo=path.name,
            repo_id=repo_id,
            repo_type="dataset",
        )
        print(f"pushed {path.name}")
except Exception as e:
    print(f"HF push skipped ({type(e).__name__}): {e}")

## 12. Summary

In [ ]:
print("Tahoe per-compound transcriptomic build summary")
print("-" * 50)
print(f"HVGs picked:         {len(hvg_set):>5d}  (baseMean > 10, top by LFC std)")
print(f"Tahoe-pooled:        {pooled.shape[0]:>5d} drugs x {pooled.shape[1]:>5d} HVGs")
print(f"Tahoe-A549:          {a549.shape[0]:>5d} drugs x {a549.shape[1]:>5d} HVGs")
print()
# Note: Tahoe has multiple drug-row entries that share an InChIKey (formulation
# variants like "Ixazomib" + "Ixazomib citrate"), so drug-row counts can exceed
# unique-compound counts. Report both for clarity.
def _drug_rows(flag): return int(manifest[flag].sum())
def _unique_inchikeys(flag, key="inchikey_full"):
    return int(manifest.loc[manifest[flag], key].nunique())

print("v0 B3 arm coverage on the Tahoe side:")
print(f"  B3b primary  Tahoe x JUMP cpg0016 :  {_unique_inchikeys('in_jump_cpg0016'):>3d} unique compounds  ({_drug_rows('in_jump_cpg0016')} Tahoe drug rows)")
print(f"  B3a robust   Tahoe x rxrx3-core   :  {_unique_inchikeys('in_rxrx3_core', 'inchikey_skeleton'):>3d} unique compounds  ({_drug_rows('in_rxrx3_core')} Tahoe drug rows)")
print(f"  B3c sanity   Tahoe-A549 x CPJUMP1 :  {_unique_inchikeys('in_jt1_treatment'):>3d} treatments + {_unique_inchikeys('in_jt1_poscon')} poscon")